# 21 — Desain dan preflight replikasi 20 seed

Run All memeriksa input, fold, jadwal dan jumlah evaluasi. Tidak melakukan training, inference, intervensi atau bootstrap. Target tetap 20; registrasi publik belum dilakukan. Baca PROTOCOL.md dan README.md sebelum notebook 22/23.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from hashlib import sha256
from contextlib import contextmanager
from importlib.metadata import version
import csv, gzip, json, os, pickle, platform, socket, sys, time
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.spatial.distance import jensenshannon
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p/'src/e2_checkpoints.py').is_file())
sys.path.insert(0, str(ROOT/'src'))
from audit_e0_e1 import AlignmentError, cache_manifest, file_sha256
from e2_checkpoints import spec_digest
PLAN = ROOT/'plans/attention_audit_replication20_20261019'
LOCK = json.loads((PLAN/'design_lock.json').read_text())
C = LOCK['configuration']
if file_sha256(PLAN/'PROTOCOL.md') != LOCK['protocol_sha256']:
    raise AlignmentError('protocol changed after design lock; rebuild before new outcomes')
for relative, expected in LOCK['inputs'].items():
    if file_sha256(ROOT/relative) != expected:
        raise AlignmentError('pinned input/source changed: '+relative)
if file_sha256(PLAN/'directed_seed_schedule.csv') != LOCK['schedule_sha256']:
    raise AlignmentError('transfer schedule differs from design lock')
schedule_all = pd.read_csv(PLAN/'directed_seed_schedule.csv')
schedule_all['primary_pair'] = schedule_all.primary_pair.astype(str).str.lower().map({'true': True, 'false': False})
if schedule_all.primary_pair.isna().any(): raise AlignmentError('invalid schedule boolean')


def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name+'.tmp.'+str(os.getpid()))
    tmp.write_text(json.dumps(value, indent=2, default=str), encoding='utf-8')
    os.replace(tmp, path)


def atomic_pickle(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name+'.tmp.'+str(os.getpid()))
    blob=gzip.compress(pickle.dumps(value,protocol=5),compresslevel=3)
    with tmp.open('wb') as stream:pickle.dump(dict(format='gzip_pickle_v1',sha256=sha256(blob).hexdigest(),payload=blob),stream,protocol=5)
    os.replace(tmp, path)


def read_local_pickle(path):
    with Path(path).open('rb') as stream:envelope=pickle.load(stream)
    if envelope['format']!='gzip_pickle_v1' or sha256(envelope['payload']).hexdigest()!=envelope['sha256']:
        raise AlignmentError('cached payload checksum mismatch: '+str(path))
    return pickle.loads(gzip.decompress(envelope['payload']))


def notebook_code_digest(path):
    doc=json.loads(Path(path).read_text())
    chunks=[]
    for cell in doc['cells']:
        if cell['cell_type']=='code':
            value=cell['source'];chunks.append(''.join(value) if isinstance(value,list) else value)
    return sha256('\n'.join(chunks).encode()).hexdigest()


def atomic_csv(path, data):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name+'.tmp.'+str(os.getpid()))
    data.to_csv(tmp, index=False); os.replace(tmp, path)


@contextmanager
def stage_lock(directory, stage):
    path = directory/(stage+'.active.lock')
    directory.mkdir(parents=True, exist_ok=True)
    if path.exists():
        owner = json.loads(path.read_text())
        if owner['host'] != socket.gethostname():
            raise RuntimeError('stage lock belongs to another host; inspect before recovering: '+str(path))
        try: os.kill(int(owner['pid']), 0)
        except ProcessLookupError:
            path.rename(path.with_name(path.name+'.stale.'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')))
        else:
            raise RuntimeError('stage already running in PID '+str(owner['pid'])+'; no concurrent Run All')
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_EXCL)
    with os.fdopen(fd, 'w') as stream:
        json.dump(dict(pid=os.getpid(), host=socket.gethostname()), stream)
    try: yield
    except BaseException as error:
        if 'manifest' in globals() and 'OUT' in globals():
            manifest.update(completion_status='interrupted' if isinstance(error,KeyboardInterrupt) else 'failed',
                            failed_stage=stage,error_type=type(error).__name__,error_message=str(error))
            atomic_json(OUT/'analysis_manifest.json',manifest)
        raise
    finally:
        if path.exists(): path.unlink()


def prepare_cohort(cohort):
    cache = ROOT/C[cohort+'_cache']
    records = cache_manifest(cache, cohort, C['patch'], C['fs'])
    if cohort == 'uci395':
        records = records.loc[records.task == C['uci_primary_task']].copy()
    records = records.rename(columns={'idx': 'cache_idx'}).reset_index(drop=True)
    records['idx'] = np.arange(len(records))
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    splits = list(splitter.split(np.zeros(len(records)), records.label, records.subject))
    folds = np.full(len(records), -1, int)
    for fold, (train, test) in enumerate(splits):
        if set(records.subject.iloc[train]) & set(records.subject.iloc[test]):
            raise AlignmentError('subject leakage in fold')
        if set(records.label.iloc[test]) != {0, 1}:
            raise AlignmentError('held-out fold lacks a class; no automatic split change')
        folds[test] = fold
    if (folds < 0).any(): raise AlignmentError('incomplete folds')
    records['fold'] = folds
    if cohort == 'newhandpd':
        old = pd.read_csv(ROOT/C['discovery_run']/'record_manifest.csv')
        if len(old) != len(records) or not np.array_equal(old.record_id, records.record_id):
            raise AlignmentError('NewHandPD records differ from discovery bank')
        field = 'fold_actual' if 'fold_actual' in old else 'fold'
        if not np.array_equal(old[field].to_numpy(int), folds):
            raise AlignmentError('NewHandPD fold identities differ')
        if records.subject.nunique() != 66 or len(records) != 264:
            raise AlignmentError('NewHandPD bank denominator differs')
    return records, splits

In [2]:
DESIGN_REPORT=PLAN/'preflight'
DESIGN_REPORT.mkdir(exist_ok=True)
budgets=[]
for cohort in ('newhandpd','uci395'):
    records,splits=prepare_cohort(cohort)
    atomic_csv(DESIGN_REPORT/(cohort+'_record_manifest.csv'),records)
    rows=[]
    for fold,(train,test) in enumerate(splits):
        for role,indices in [('train',train),('test',test)]:
            for idx in indices:rows.append(dict(fold=fold,role=role,idx=int(idx),
                record_id=records.iloc[idx].record_id,subject=records.iloc[idx].subject))
    atomic_csv(DESIGN_REPORT/(cohort+'_fold_assignments.csv'),pd.DataFrame(rows))
    for arm in C['architectures']:
        graph=schedule_all.loc[(schedule_all.cohort==cohort)&(schedule_all.architecture==arm)]
        edges=set(zip(graph.target_seed,graph.source_seed))
        if len(graph)!=100 or not all((b,a) in edges and a!=b for a,b in edges):
            raise AlignmentError('schedule uniqueness/reciprocity failure')
        if not (graph.groupby('target_seed').size()==5).all() or not (graph.groupby('source_seed').size()==5).all():
            raise AlignmentError('schedule degree imbalance')
        primary=graph.loc[graph.primary_pair]
        # Bounds: no cache reuse, all search quotas succeed, both references pass.
        # Per rank: 4 budgets * (median 1+20 forwards + donor 5+20 forwards).
        rank_bound=1+5+2*5
        forward_bound=len(records)*20*rank_bound*4*(21+25)
        budgets.append(dict(cohort=cohort,architecture=arm,n_records=len(records),
            n_subjects=records.subject.nunique(),n_HC=records.loc[records.label==0].subject.nunique(),
            n_PD=records.loc[records.label==1].subject.nunique(),n_checkpoints=100,
            new_checkpoints=85 if cohort=='newhandpd' else 100,
            scheduled_directions=len(graph),primary_directions=len(primary),
            scheduled_pair_replacement_rows=len(records)*len(graph)*2,
            primary_pair_replacement_rows=len(records)*len(primary)*2,
            no_reuse_successful_quota_forward_upper_bound=forward_bound))
budget=pd.DataFrame(budgets)
atomic_csv(DESIGN_REPORT/'planned_budget.csv',budget)
atomic_json(DESIGN_REPORT/'completion.json',dict(completion_status='complete',
    design_id=LOCK['design_id'],protocol_sha256=LOCK['protocol_sha256'],
    training_performed=False,model_inference_performed=False,new_interventions_performed=False,
    note='Input/split/design checks only; no empirical effect, runtime or precision estimate.'))
display(budget)
print('PREFLIGHT COMPLETE. No model has run. Read the budget before notebooks 22/23.')
print('Forward counts are conservative design bounds, not actual work or elapsed time.')

,cohort,architecture,n_records,n_subjects,n_HC,n_PD,n_checkpoints,new_checkpoints,scheduled_directions,primary_directions,scheduled_pair_replacement_rows,primary_pair_replacement_rows,no_reuse_successful_quota_forward_upper_bound
0,newhandpd,gru,264,66,35,31,100,85,100,70,52800,36960,15544320
1,newhandpd,mamba2,264,66,35,31,100,85,100,76,52800,40128,15544320
2,uci395,gru,77,76,15,61,100,100,100,100,15400,15400,4533760
3,uci395,mamba2,77,76,15,61,100,100,100,100,15400,15400,4533760


PREFLIGHT COMPLETE. No model has run. Read the budget before notebooks 22/23.
Forward counts are conservative design bounds, not actual work or elapsed time.
